# AI Guide: demo YOLO + SegFormer trên video

Notebook này chạy YOLO vật cản và SegFormer phân đoạn lớp `sidewalk` trên cùng từng khung hình, dùng GPU Kaggle nếu có. Nó xuất video có hộp nhận diện, vùng sidewalk màu xanh và chú thích cảnh báo tương đối. Đây là demo nghiên cứu; không phải công cụ dẫn đường an toàn.

## Chuẩn bị dữ liệu đầu vào
Tạo một Kaggle Dataset riêng, tải lên model YOLO `best.pt`, thư mục `segformer_sidewalk` (có `config.json`, `model.safetensors`, `preprocessor_config.json`) và một video `.mp4`. Gắn dataset vào notebook bằng **Add Input**. Có thể gắn nhiều input; ô dò file bên dưới sẽ liệt kê và chọn file tìm thấy đầu tiên.

Trong **Settings → Accelerator**, chọn GPU nếu quota tài khoản còn. Bản notebook không mở Tkinter, camera laptop hoặc TTS Windows; nó xử lý video đã tải lên.

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "transformers==5.20.0" safetensors

In [ ]:
from pathlib import Path
import json, time, csv
import cv2
import numpy as np
import torch
import transformers
from ultralytics import YOLO
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
from IPython.display import display, Video

INPUT_ROOT = Path('/kaggle/input')
YOLO_CANDIDATES = sorted(p for p in INPUT_ROOT.rglob('*.pt') if p.name in {'best.pt', 'yolo26n_best.pt'})
SEGFORMER_DIRS = []
for cfg_path in INPUT_ROOT.rglob('config.json'):
    try:
        cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    except Exception:
        continue
    if cfg.get('model_type') == 'segformer' and any((cfg_path.parent / n).is_file() for n in ('model.safetensors', 'pytorch_model.bin')):
        SEGFORMER_DIRS.append(cfg_path.parent)
SEGFORMER_DIRS = sorted(set(SEGFORMER_DIRS))
VIDEO_CANDIDATES = sorted(p for p in INPUT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv'})

print('CUDA available:', torch.cuda.is_available())
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__)
print('YOLO candidates:')
for p in YOLO_CANDIDATES: print(' -', p)
print('SegFormer folders:')
for p in SEGFORMER_DIRS: print(' -', p)
print('Videos:')
for p in VIDEO_CANDIDATES: print(' -', p)

assert YOLO_CANDIDATES, 'Add the trained best.pt as a Kaggle Dataset input.'
assert SEGFORMER_DIRS, 'Add the complete SegFormer model folder as a Kaggle Dataset input.'
assert VIDEO_CANDIDATES, 'Add at least one demo video as a Kaggle Dataset input.'

# If several inputs were found, replace these with a listed path.
YOLO_PATH = YOLO_CANDIDATES[0]
SEGFORMER_PATH = SEGFORMER_DIRS[0]
VIDEO_PATH = VIDEO_CANDIDATES[0]
OUTPUT_VIDEO = Path('/kaggle/working/ai_guide_demo.mp4')
OUTPUT_CSV = Path('/kaggle/working/ai_guide_events.csv')
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
USE_CUDA = DEVICE.startswith('cuda')
print('\nSelected:', YOLO_PATH, '|', SEGFORMER_PATH, '|', VIDEO_PATH, '| device:', DEVICE)

In [ ]:
detector = YOLO(str(YOLO_PATH))
processor = AutoImageProcessor.from_pretrained(str(SEGFORMER_PATH), local_files_only=True)
segmenter = SegformerForSemanticSegmentation.from_pretrained(str(SEGFORMER_PATH), local_files_only=True).to(DEVICE).eval()

labels = {int(k): str(v).strip().lower().replace('_', ' ') for k, v in segmenter.config.id2label.items()}
walkable_names = {'sidewalk', 'walkway', 'footpath', 'pedestrian path', 'pedestrian walkway'}
walkable_ids = [i for i, name in labels.items() if name in walkable_names]
assert walkable_ids, f'No sidewalk class found in model id2label. Labels: {list(labels.values())[:20]}'
print('Walkable labels:', [(i, labels[i]) for i in walkable_ids])
print('SegFormer classes:', segmenter.config.num_labels)
print('YOLO classes:', detector.names)

def sidewalk_mask(frame_bgr, threshold=0.45):
    h, w = frame_bgr.shape[:2]
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    inputs = processor(images=rgb, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
            logits = segmenter(**inputs).logits
        logits = torch.nn.functional.interpolate(logits.float(), size=(h, w), mode='bilinear', align_corners=False)
        probs = logits.softmax(dim=1)
        pred = probs.argmax(dim=1)[0]
        target_prob = probs[:, walkable_ids].amax(dim=1)[0]
        selected = torch.zeros_like(pred, dtype=torch.bool)
        for class_id in walkable_ids: selected |= (pred == class_id)
        mask = selected & (target_prob >= threshold)
    return mask.to('cpu').numpy().astype(np.uint8)

print('Models loaded on', DEVICE)

In [ ]:
# Demo settings: process the first 20 seconds; increase MAX_SECONDS for a longer clip.
MAX_SECONDS = 20
YOLO_IMGSZ = 480
YOLO_CONF = 0.30
MAX_SIDE = 960
CONTEXT_CLASSES = {'building', 'road', 'traffic sign', 'pedestrian crosswalk'}
records, latencies_ms = [], []
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened(): raise RuntimeError(f'Could not open video: {VIDEO_PATH}')
source_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_limit = int(source_fps * MAX_SECONDS) if MAX_SECONDS else int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
writer = None
frame_index = 0
start_all = time.perf_counter()

while frame_index < frame_limit:
    ok, frame = cap.read()
    if not ok: break
    if max(frame.shape[:2]) > MAX_SIDE:
        scale = MAX_SIDE / max(frame.shape[:2])
        frame = cv2.resize(frame, (round(frame.shape[1]*scale), round(frame.shape[0]*scale)), interpolation=cv2.INTER_AREA)
    h, w = frame.shape[:2]
    if writer is None:
        writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*'mp4v'), source_fps, (w, h))
        if not writer.isOpened(): raise RuntimeError('Could not create output mp4.')

    t0 = time.perf_counter()
    mask = sidewalk_mask(frame)
    pred = detector.predict(frame, imgsz=YOLO_IMGSZ, conf=YOLO_CONF, device=0 if USE_CUDA else 'cpu', quantize=16 if USE_CUDA else 32, verbose=False)[0]
    latency = (time.perf_counter() - t0) * 1000
    latencies_ms.append(latency)

    overlay = frame.copy()
    overlay[mask > 0] = (40, 180, 40)
    rendered = cv2.addWeighted(overlay, 0.25, frame, 0.75, 0) if np.any(mask) else frame.copy()
    blocked = np.zeros((h, w), dtype=np.uint8)
    hazards = []
    if pred.boxes is not None:
        for box in pred.boxes:
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            cls_id = int(box.cls.item())
            name = str(pred.names[cls_id])
            norm_name = name.strip().lower().replace('_', ' ')
            score = float(box.conf.item())
            if norm_name not in CONTEXT_CLASSES:
                pad = max(5, round(w * 0.02))
                blocked[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)] = 1
                if y2 >= h * 0.52 and x2 >= w * 0.16 and x1 <= w * 0.84:
                    bh = max(0, y2-y1)
                    hr, br = bh/max(h,1), y2/max(h,1)
                    band = 'near' if hr >= .40 or (hr >= .27 and br >= .92) else 'medium' if hr >= .17 or br >= .80 else 'far'
                    cx = (x1+x2)/2/max(w,1)
                    direction = 'left' if cx < .38 else 'right' if cx > .62 else 'ahead'
                    hazards.append((band, score, name, direction, (x1,y1,x2,y2)))
                    color = (0, 70, 255) if band == 'near' else (0, 190, 255) if band == 'medium' else (255, 180, 0)
                    cv2.rectangle(rendered, (x1,y1), (x2,y2), color, 2)
                    cv2.putText(rendered, f'{name} {band} {score:.2f}', (max(0,x1), max(18,y1-7)), cv2.FONT_HERSHEY_SIMPLEX, .55, color, 2, cv2.LINE_AA)

    corridors = []
    for side, left, right in [('left', .15, .38), ('ahead', .38, .62), ('right', .62, .85)]:
        y0, y1 = int(h*.60), int(h*.96)
        x0, x1 = int(w*left), int(w*right)
        region = mask[y0:y1, x0:x1].copy()
        occupied = blocked[y0:y1, x0:x1]
        if region.size and not np.any(occupied) and region.mean() >= .85:
            count, components, stats, _ = cv2.connectedComponentsWithStats(region, 8)
            if count > 1:
                largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
                connected = components == largest
                edge = max(1, len(region)//8)
                if connected[:edge].mean() > .8 and connected[-edge:].mean() > .8:
                    corridors.append(side)

    if hazards:
        order = {'near': 0, 'medium': 1, 'far': 2}
        band, score, name, direction, _ = min(hazards, key=lambda item: (order[item[0]], -item[1]))
        caption = f'STOP: {name} {band} ahead' if band == 'near' and direction == 'ahead' else f'SLOW: {name} {band} on {direction}'
        records.append({'frame': frame_index, 'time_s': round(frame_index/max(source_fps,1),2), 'class': name, 'confidence': round(score,3), 'direction': direction, 'distance_band': band, 'clear_corridors': '|'.join(corridors)})
    else:
        caption = 'No detected obstacle; path is not verified safe'
    cv2.rectangle(rendered, (0,0), (w,34), (20,30,40), -1)
    cv2.putText(rendered, caption[:100], (10,23), cv2.FONT_HERSHEY_SIMPLEX, .58, (255,255,255), 2, cv2.LINE_AA)
    writer.write(rendered)
    frame_index += 1

cap.release()
if writer is not None: writer.release()
elapsed = time.perf_counter() - start_all
print(f'Frames processed: {frame_index}; source FPS: {source_fps:.1f}; wall time: {elapsed:.1f}s')
if latencies_ms:
    print(f'Combined inference FPS: {len(latencies_ms)/sum(latencies_ms)*1000:.2f}; p50: {np.percentile(latencies_ms,50):.1f} ms; p95: {np.percentile(latencies_ms,95):.1f} ms')
if OUTPUT_VIDEO.exists(): print('Video:', OUTPUT_VIDEO, '| MB:', round(OUTPUT_VIDEO.stat().st_size/1e6,2))
if records:
    with OUTPUT_CSV.open('w', newline='', encoding='utf-8-sig') as f:
        writer_csv = csv.DictWriter(f, fieldnames=records[0].keys())
        writer_csv.writeheader(); writer_csv.writerows(records)
    print('Obstacle event table:', OUTPUT_CSV)
else:
    print('No YOLO hazards passed the confidence/position filters in this clip.')

In [ ]:
if OUTPUT_VIDEO.exists():
    display(Video(str(OUTPUT_VIDEO), embed=True, html_attributes='controls muted'))
if OUTPUT_CSV.exists():
    import pandas as pd
    display(pd.read_csv(OUTPUT_CSV).head(30))

## Đọc kết quả
- Xanh lá là pixel SegFormer dự đoán thuộc lớp `sidewalk`; hộp màu là vật thể YOLO. Hộp chặn hành lang trong cùng vùng ảnh.
- `near/medium/far` chỉ là ước lượng tương đối từ kích thước hộp, không phải mét. Caption `STOP/SLOW` là minh họa thị giác; notebook không phát giọng nói.
- FPS và p95 ở trên đo thời gian suy luận hai model, không bao gồm camera điện thoại, giao diện desktop hoặc âm thanh.
- Model SegFormer hiện tại có 150 lớp ADE20K. Kết quả cần được đối chiếu với video ngoài trời; không coi mặt nạ màu xanh là chứng nhận lối đi an toàn.

Tệp xuất nằm trong `/kaggle/working/ai_guide_demo.mp4` và `/kaggle/working/ai_guide_events.csv`; tải chúng từ **Output** hoặc lưu phiên bằng **Save Version**.